# Does earlier director rating history help?

One shared, fixed-settings **Ridge** experiment replaces the repeated version in the model notebooks. Compare all available predictors with versus without the three director-rating-history variables. Existing financial-history predictors remain in both versions.

Use the same earlier training movies and chronological folds. No test data choose features; this notebook reports validation only. Results concern this representation and estimator, not every possible model. No CSVs or app artifacts are changed.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../Modeling_scripts').resolve()))
from model_workflow import load_data, compare_director_rating_history
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import Ridge

DATA_FILE = '../data/model_data_rating.csv'
CUTOFF_DATE = '2017-11-18'
N_VALIDATION_SPLITS = 3
MODEL_SETTINGS = {'alpha': 10.0}
data = load_data(DATA_FILE, 'rating', CUTOFF_DATE, N_VALIDATION_SPLITS)

13,276 movies | 9,405 training / 3,871 later test | 40 available predictors


In [2]:
def make_model(columns):
    """Fresh preprocessing and regression for each feature set and split."""
    numeric = [column for column in data["NUMERIC_FEATURES"] if column in columns]
    categorical = [column for column in data["CATEGORICAL_FEATURES"] if column in columns]
    binary = [column for column in data["BINARY_FEATURES"] if column in columns]
    assert set(numeric + categorical + binary) == set(columns)
    preprocessing = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("polynomial", PolynomialFeatures(degree=1, include_bias=False)),
            ("expanded_scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("binary", "passthrough", binary),
    ])
    return Pipeline([
        ("preprocessor", preprocessing),
        ("regression", Ridge(**MODEL_SETTINGS)),
    ])

In [3]:
fold_results = compare_director_rating_history(make_model, data)
summary = fold_results.groupby('Version', sort=False)[['MAE', 'RMSE', 'R2']].mean()
display(summary.round(4))
print('Positive MAE reduction favors adding director rating history:',
      round(summary.iloc[0].MAE - summary.iloc[1].MAE, 4))

,MAE,RMSE,R2
Version,,,
Without director rating history,0.5217,0.6700,0.3083
With director rating history,0.5209,0.6661,0.3163


Positive MAE reduction favors adding director rating history: 0.0009


These are cumulative feature comparisons on reused validation folds, not causal effects or independent evaluation. Director ratings are retrospective TMDB snapshots linked to IMDb identities. Coverage is incomplete, and supporting counts/no-history flags are included with the smoothed rating feature.